
# 📘 Pix2Struct Fine-Tuning avec LoRA 



## 1. Vérification Runtime / GPU

In [ ]:

# [1] Runtime / GPU
import sys, platform, torch
print("Python:", sys.version.split()[0])
print("OS:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA dispo:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 2. Installations 

In [ ]:

# [2] Installations
# NOTE: vous pouvez commenter des libs si déjà présentes.
!pip -q install --upgrade pip
!pip -q install pdf2image pillow
!pip -q install transformers>=4.41.0 datasets accelerate torchvision huggingface_hub
!pip -q install peft bitsandbytes evaluate
# Poppler pour pdf2image (Colab/Debian)
!apt-get -y -qq install poppler-utils || true
print("✅ Dépendances installées")


## 3. Configuration des chemins & options

In [ ]:

# [3] Configuration
from pathlib import Path

# Pour Colab + Drive : mettez USE_COLAB=True et montez Drive dans l'étape suivante
USE_COLAB = True
BASE_DIR = Path("/content/drive/MyDrive/pix2struct_project")  # changez si besoin

# Dossiers d'I/O
PDF_DIR = BASE_DIR / "reports"                 # PDFs d'entrée
PDF_IMAGES_DIR = BASE_DIR / "reports_page1"    # PNG de la page 1
TRAINING_DATA_DIR = BASE_DIR / "training_data" # {train,val,test}/ *.png|*.jpg
ANNOTATIONS_DIR = BASE_DIR / "annotations_data"# {train,val,test}/ *.json|*.txt
OUTPUT_MODEL_DIR = BASE_DIR / "pix2struct-finetuned"

# Fichiers d'annotations agrégées
TRAIN_JSON = TRAINING_DATA_DIR / "train_annotations.json"
VAL_JSON   = TRAINING_DATA_DIR / "val_annotations.json"
TEST_JSON  = TRAINING_DATA_DIR / "test_annotations.json"

# Pipeline switches
DO_PDF_TO_IMAGES = False   # True pour convertir PDF -> PNG (page 1)
DO_BUILD_ANNOTS  = True    # True pour (re)générer {train,val,test}_annotations.json
USE_LORA         = True    # LoRA param-efficient finetune
DO_TRAIN         = True    # Lancer l'entraînement
DO_EVAL          = True    # Évaluer rapidement (Exact Match)

for p in [PDF_DIR, PDF_IMAGES_DIR, TRAINING_DATA_DIR, ANNOTATIONS_DIR, OUTPUT_MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("✅ Config OK :", BASE_DIR)


## 4.  Monter Google Drive

In [ ]:

# [4] Monter Drive
def in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

if USE_COLAB and in_colab():
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=True)
    print("✅ Google Drive monté")
else:
    print("⏭️ Skip mount (local)")


## 5. (Optionnel) Conversion PDF → Image (page 1)

In [ ]:

# [5] PDF -> PNG (page 1)
from pathlib import Path

if DO_PDF_TO_IMAGES:
    from pdf2image import convert_from_path
    PDF_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
    pdf_files = sorted([p for p in Path(PDF_DIR).glob("*.pdf")])
    print(f"PDFs trouvés: {len(pdf_files)}")
    for pdf in pdf_files:
        try:
            images = convert_from_path(str(pdf), first_page=1, last_page=1, dpi=200)
            if images:
                out = PDF_IMAGES_DIR / (pdf.stem + ".png")
                images[0].save(out)
                print("✅", out)
        except Exception as e:
            print("⚠️ Erreur PDF -> image:", pdf.name, e)
else:
    print("⏭️ Skip conversion PDF")


## 6. Génération des annotations agrégées (train/val/test)

In [ ]:

# [6] Build annotations
import json
from pathlib import Path

def read_label_for_image(split_dir: Path, annotations_dir: Path, img_path: Path):
    stem = img_path.stem
    cand_json = annotations_dir / split_dir.name / f"{stem}.json"
    cand_txt  = annotations_dir / split_dir.name / f"{stem}.txt"
    if cand_json.exists():
        try:
            data = json.loads(cand_json.read_text(encoding="utf-8"))
            for key in ("text", "label", "answer"):
                if key in data and isinstance(data[key], str):
                    return data[key].strip()
        except Exception as e:
            print("⚠️ JSON invalide:", cand_json.name, e)
    if cand_txt.exists():
        try:
            return cand_txt.read_text(encoding="utf-8").strip().splitlines()[0]
        except Exception as e:
            print("⚠️ TXT invalide:", cand_txt.name, e)
    return None

def build_annotations(training_data_dir: Path, annotations_dir: Path, out_path: Path):
    split = out_path.stem.replace("_annotations","")  # train_annotations -> train
    split_dir = training_data_dir / split
    if not split_dir.exists():
        print(f"⏭️ Dossier split '{split}' introuvable, skip.")
        return
    items = []
    images = sorted(split_dir.glob("*.png")) + sorted(split_dir.glob("*.jpg"))
    for img in images:
        label = read_label_for_image(split_dir, annotations_dir, img)
        if label is None:
            print("⚠️ Pas de label pour", img.name, "— ignoré.")
            continue
        items.append({"image": img.name, "text": label})
    if items:
        out_path.write_text(json.dumps(items, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"✅ {out_path.name} écrit ({len(items)} items).")
    else:
        print(f"⚠️ Aucun item pour {split}.")

if DO_BUILD_ANNOTS:
    for path in (TRAIN_JSON, VAL_JSON, TEST_JSON):
        build_annotations(TRAINING_DATA_DIR, ANNOTATIONS_DIR, path)
else:
    print("⏭️ Skip build annotations")


## 7. Chargement Dataset & Processor Pix2Struct

In [ ]:

# [7] Dataset & Processor
from datasets import Dataset, DatasetDict
from PIL import Image
import json
from transformers import Pix2StructProcessor

MODEL_ID = "google/pix2struct-docvqa-large"
processor = Pix2StructProcessor.from_pretrained(MODEL_ID)

def load_split(split_json):
    if not split_json.exists():
        return None
    data = json.loads(split_json.read_text(encoding="utf-8"))
    return Dataset.from_list(data)

raw = {}
for split, path in [("train", TRAIN_JSON), ("validation", VAL_JSON), ("test", TEST_JSON)]:
    ds = load_split(path)
    if ds is not None and len(ds) > 0:
        raw[split] = ds

dataset = DatasetDict(raw) if raw else None
print(dataset)


## 8. Préprocessing (images+textes → features)

In [ ]:

# [8] Preprocess
from functools import partial
from PIL import Image

def preprocess_batch(examples, split_name):
    images, texts = [], []
    base = TRAINING_DATA_DIR / split_name
    for img_name, text in zip(examples["image"], examples["text"]):
        img_path = base / img_name
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            print("⚠️ Image manquante:", img_path.name, e)
            from PIL import Image as PILImage
            img = PILImage.new("RGB", (384, 384), (255,255,255))
        images.append(img)
        texts.append(text)

    model_inputs = processor(images=images, text=texts, padding="max_length", max_patches=2048, return_tensors="pt")
    labels = processor.tokenizer(texts, padding="max_length", truncation=True, return_tensors="pt").input_ids
    labels[labels == processor.tokenizer.pad_token_id] = -100
    model_inputs["labels"] = labels
    return {k: v.tolist() if hasattr(v, "tolist") else v for k, v in model_inputs.items()}

if dataset is not None:
    proc = {}
    for split in dataset.keys():
        proc[split] = dataset[split].map(partial(preprocess_batch, split_name=split), batched=True, remove_columns=dataset[split].column_names)
    dataset = DatasetDict(proc)
    print(dataset)
else:
    print("⚠️ Dataset vide — vérifiez vos chemins et annotations.")


## 9. Entraînement (LoRA optionnel)

In [ ]:

# [9] Training
import torch
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, default_data_collator
from transformers import Pix2StructForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType

if dataset is None or "train" not in dataset:
    print("⚠️ Rien à entraîner (split 'train' absent).")
else:
    model = Pix2StructForConditionalGeneration.from_pretrained(MODEL_ID)
    model.config.forced_decoder_ids = None  # important pour génération libre

    if USE_LORA:
        try:
            lora_config = LoraConfig(
                r=16,
                lora_alpha=32,
                lora_dropout=0.05,
                bias="none",
                task_type=TaskType.SEQ_2_SEQ_LM,
                target_modules=["q","k","v","o","q_proj","k_proj","v_proj","o_proj"],
            )
            model = get_peft_model(model, lora_config)
            print("✅ LoRA appliqué.")
        except Exception as e:
            print("⚠️ LoRA non appliqué, fallback FT standard:", e)

    training_args = Seq2SeqTrainingArguments(
        output_dir=str(OUTPUT_MODEL_DIR / "checkpoints"),
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        learning_rate=5e-5,
        num_train_epochs=1,
        fp16=torch.cuda.is_available(),
        gradient_accumulation_steps=4,
        save_strategy="epoch",
        evaluation_strategy="epoch" if "validation" in dataset else "no",
        logging_steps=50,
        predict_with_generate=True,
        report_to="none",
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset.get("train"),
        eval_dataset=dataset.get("validation"),
        data_collator=default_data_collator,
    )

    if DO_TRAIN:
        trainer.train()
        print("✅ Entraînement terminé.")

    # Sauvegarde modèle/processor
    if DO_TRAIN:
        OUTPUT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(OUTPUT_MODEL_DIR)
        processor.save_pretrained(OUTPUT_MODEL_DIR)
        print("✅ Modèle & processor sauvegardés dans:", OUTPUT_MODEL_DIR)


## 10. Évaluation rapide (Exact Match approximatif)

In [ ]:

# [10] Eval
import torch
from transformers import Pix2StructForConditionalGeneration

def exact_match(a: str, b: str) -> bool:
    return a.strip().lower() == b.strip().lower()

if DO_EVAL and dataset is not None and "validation" in dataset:
    model_path = OUTPUT_MODEL_DIR if (OUTPUT_MODEL_DIR / "config.json").exists() else MODEL_ID
    model = Pix2StructForConditionalGeneration.from_pretrained(model_path)
    model.eval().to("cuda" if torch.cuda.is_available() else "cpu")

    n = min(20, len(dataset["validation"]))
    correct = 0
    for i in range(n):
        item = dataset["validation"][i]
        import numpy as np
        inputs = {k: torch.tensor(item[k]).unsqueeze(0).to(model.device) for k in ("flattened_patches", "attention_mask") if k in item}
        with torch.no_grad():
            gen = model.generate(**inputs, max_new_tokens=64)
        pred = processor.tokenizer.decode(gen[0], skip_special_tokens=True)
        gt_ids = torch.tensor(item["labels"])
        gt_ids = torch.where(gt_ids==-100, torch.tensor(processor.tokenizer.pad_token_id), gt_ids)
        gt = processor.tokenizer.decode(gt_ids, skip_special_tokens=True)
        if exact_match(pred, gt):
            correct += 1

    print(f"Exact Match (approx) sur {n} échantillons: {correct}/{n} = {correct/max(1,n):.2%}")
else:
    print("⏭️ Skip évaluation")


## 11. Inférence sur une image

In [ ]:

# [11] Inference
from PIL import Image
import torch
from transformers import Pix2StructForConditionalGeneration

IMAGE_FOR_INFER = None  # ex: TRAINING_DATA_DIR / "test" / "example.png"

if IMAGE_FOR_INFER:
    model_path = OUTPUT_MODEL_DIR if (OUTPUT_MODEL_DIR / "config.json").exists() else MODEL_ID
    model = Pix2StructForConditionalGeneration.from_pretrained(model_path)
    model.to("cuda" if torch.cuda.is_available() else "cpu").eval()
    img = Image.open(IMAGE_FOR_INFER).convert("RGB")

    inputs = processor(images=[img], text=[""], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=64)
    print("🔮 Prediction:", processor.tokenizer.decode(out[0], skip_special_tokens=True))
else:
    print("ℹ️ Définissez IMAGE_FOR_INFER pour tester une image.")


## 12. (Optionnel) Générer la structure d'exemple des dossiers

In [ ]:

# [12] Scaffolding exemple
for split in ("train","validation","test"):
    (TRAINING_DATA_DIR / split).mkdir(parents=True, exist_ok=True)
    (ANNOTATIONS_DIR / split).mkdir(parents=True, exist_ok=True)
print("✅ Arborescence créée sous", BASE_DIR)
print("Placez vos images dans training_data/{train,validation,test} et vos labels dans annotations_data/{train,validation,test}.")
